# Homework: Agentic RAG

## Question 1: How many lesson pages

> How many lesson pages are in the dataset?
>
> * 24
> * 72
> * 240
> * 720

The instructions give the code below that can traverse files in a Github repo:

In [32]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

With this code it's simple to find the number of pages:

In [33]:
files = reader.read()
print("Q1. How many lesson pages?")
print("Answer: {}".format(len(files)))  

Q1. How many lesson pages?
Answer: 72


## Question 2: Indexing and searching


> Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:
>
>>     How does the agentic loop keep calling the model until it stops?
>
> What's the filename of the first result?
>
>    * 01-agentic-rag/lessons/03-rag.md
>    * 01-agentic-rag/lessons/14-agentic-loop.md
>    * 04-evaluation/lessons/13-llm-as-judge.md
>    * 06-best-practices/lessons/02-hybrid-search.md


In [34]:
# Prepare the lesson markdown documents for indexing
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [35]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

results = index.search("How does the agentic loop keep calling the model until it stops?")

print("Q2: What's the filename of the first result?")
print("Answer: {}".format(results[0]["filename"]))

Q2: What's the filename of the first result?
Answer: 01-agentic-rag/lessons/14-agentic-loop.md


## Question 3: RAG

> Build a RAG over the index from Q2 and answer the query:
>
>>     How does the agentic loop keep calling the model until it stops?
>
> Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?
>
>   * 700
>   * 7000
>   * 70000
>   * 700000


I modified the provided `RAGBase` class from [rag_helper.py](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/code/rag_helper.py) slightly:
* Renamed to ZoomCampRAG
* Requires the `INSTRUCTION` and `PROMPT_TEMPLATE` to be passed into the constructor because it feels like they should be independent of the helper class
* Renamed method `rag` to `query` and 'privatized' other methods by prefixing with `__` since only `query` seems like it should be externally callable
* Commented out the `boost_dict` and `filter_dict` because they require the names of the index data fields and with only `filename` and `content` there isn't likely much benefit to boosting.

Other changes that might be useful:

Allow `boost_dict` and `filter_dict` to be passed in to the `query` method because they're more closely tied with the query string than to the RAG helper itself.

With the `ZoomCampRAG` class and the indexed data from earlier, we can use the INSTRUCTION and PROMPT_TEMPLATE values from the lesson and answer this question.

In [36]:
from dotenv import load_dotenv
from openai import OpenAI
from ZoomCampRAG import ZoomCampRAG

load_dotenv()
openai_client = OpenAI()


INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

question3 = "How does the agentic loop keep calling the model until it stops?"

assistant = ZoomCampRAG(index, openai_client, INSTRUCTIONS, PROMPT_TEMPLATE)

output_text, usage = assistant.query(question3)

print("Q3: Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?")
print("Answer: {}".format(usage.input_tokens))

Q3: Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?
Answer: 6988


6988 is close enough to 7000 so that is our answer.

## Question 4: Chunking

> How many chunks do you get?
>
>    * 70
>    * 295
>    * 1100
>    * 4500


The homework instructions give us the function to do the chunking as well as the parameters to use, so we can answer this question easily by finding the length of the result.

In [37]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print ("Q4: How many chunks do you get?")
print ("Answer: {}".format(len(chunks)))

Q4: How many chunks do you get?
Answer: 295


# Question 5: RAG with chunking

> Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
>
>    * about the same
>    * 3× fewer
>    * 10× fewer
>    * 30× fewer

I think the question means 1/3, 1/10, 1/30 the number of tokens compared to the non-chunked index.

We create a new `ZoomCampRAG` object with the chunked index, invoke the query with the `question3` string and compare the token usage.

In [38]:
chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# Fit the chunked index with the chunks
chunked_index.fit(chunks)

chunked_assistant = ZoomCampRAG(chunked_index, openai_client, INSTRUCTIONS, PROMPT_TEMPLATE)

output_text, chunked_usage = chunked_assistant.query(question3)

print("Q5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?")
print("Answer: {}".format(round(usage.input_tokens / chunked_usage.input_tokens)))

Q5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
Answer: 3


## Question 6: Turning it into an agent
>
> How many times did the agent call search?
> 
>>     Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI gpt-5.4-mini; with a different model or provider the number may differ, so keep that in mind.
>
>    * about the same
>    * 0
>    * 4
>    * 10
>    * 20

In [ ]:
from openai import OpenAI

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback, OpenAIResponsesRunner

agent_tools = Tools()

# Track the number of times the search function is called. This is a bit hacky, but I don't
# understand the 'LoopResult' class well enough to programmatically get the result.
# The 'all_messages' field seems to have the prompts, the tool search results and the final response.
search_calls = 0
def search(query: str) -> dict[str, str]:
    """Search the course lesson text."""
    global search_calls
    search_calls += 1
    return chunked_index.search(
        query,
        num_results=5,
    )

agent_tools.add_tool(search)

# Create chat interface and client
chat_interface = IPythonChatInterface()

# Don't use the callback because the output is a lot and isn't needed.
# callback = DisplayingRunnerCallback(chat_interface)
openai_client = OpenAIClient(
    model="gpt-5.4-mini",
    client=OpenAI()
)

QUESTION_6_INSTRUCTIONS ='''
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
'''

question6 = 'How does the agentic loop work, and how is it different from plain RAG?'

# Create and run chat assistant
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=QUESTION_6_INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=openai_client
)

result = runner.loop(
    prompt=question6,
    # callback=callback,
)


print("Q6: How many times did the agent call search?")
print("Answer: {}".format(search_calls))

Q6: How many times did the agent call search?
Answer: 4
